# MLflow Model

## 1 Pacote de modelos

Vamos entender o funcionamento interno do pacote de modelos do MLflow, explorando a estrutura de arquivos e metadados que permitem que um modelo seja portátil e independente do código de treinamento. Essa estrutura, composta por definições de assinatura e exemplos de entrada, ela estabelece o contrato necessário para a implantação e o consumo do modelo como uma API de forma padronizada.

Vamos inspecionar os artefatos de um modelo (que registramos) via alias, comparar as diferentes formas de carregamento através dos _flavors_ `scikit-learn` e `pyfunc`, e testar a validação de entrada de dados para garantir a integridade das inferências, preparando o terreno para a criação de interfaces de predição mais ricas e personalizadas.

### 1.1 Recuperando modelo em produção

Primeiro, precisamos levantar o servidor do MLflow:

```terminal
mlflow server --backend-store-uri sqlite:///./mlflow.db --default-artifact-root ./mlartifacts --host 127.0.0.1 --port 5000
```

E depois podemos recuperar o modelo em produção, sem utilizar nenhum número para especificar a versão, apenas referenciando o modelo atualmente em produção pelo _alias_:

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

available_registered_models = mlflow.search_registered_models()
len(available_registered_models)

1

In [8]:
# temos apenas um modelo registrado
available_registered_models[0].name

'credit_risk_classifier'

In [10]:
REGISTERED_MODEL_NAME = available_registered_models[0].name # "credit_risk_classifier"
production_model_uri  = f"models:/{REGISTERED_MODEL_NAME}@production"
production_model_uri

'models:/credit_risk_classifier@production'

### 1.2 Um pacote portável

O **MLflow Model** é um formato de empacotamento padronizado que o MLflow utiliza para salvar modelos de _Machine Learning_. Diferente de salvar apenas um arquivo isolado (como um `.pkl`), o MLflow cria um diretório completo que garante a portabilidade e a reprodutibilidade do modelo. É como se fosse o "passo a passo" do que ele precisa para funcionar em qualquer máquina, dessa forma, outras ferramentas conseguem entender como carregar e executar o modelo.

Vamos acessar esse diretório e listar os arquivos disponíveis:

In [11]:
local_model_path = mlflow.artifacts.download_artifacts(artifact_uri=production_model_uri)
local_model_path

'/home/bruno/estudos/data-science-portifolio/mlflow/mlartifacts/5/models/m-493132b1b8b64441bb51b13a29ab3fe3/artifacts'

In [ ]:
from pathlib import Path

local_model_path = Path(local_model_path)
local_model_path

PosixPath('/home/bruno/estudos/data-science-portifolio/mlflow/mlartifacts/5/models/m-493132b1b8b64441bb51b13a29ab3fe3/artifacts')

In [13]:
print(local_model_path)

/home/bruno/estudos/data-science-portifolio/mlflow/mlartifacts/5/models/m-493132b1b8b64441bb51b13a29ab3fe3/artifacts


In [14]:
for path in sorted(local_model_path.rglob("*")):
    print(path.relative_to(local_model_path))

MLmodel
conda.yaml
input_example.json
model.pkl
python_env.yaml
registered_model_meta
requirements.txt
serving_input_example.json


Dentro desse diretório, temos os seguintes arquivos:

- `MLmodel`: é o principal arquivo de configuração (em formato YAML). Atua como o guia do pacote, listando os _flavors_ suportados (como `sklearn` ou `pyfunc`), indicando o caminho para o artefato do modelo, definindo as dependências necessárias para execução, a assinatura e a versão do MLflow utilizada.
- `conda.yaml`: definições do ambiente virtual Conda necessárias para rodar o modelo, lista as bibliotecas e versões específicas para garantir que o ambiente de execução seja idêntico ao de treinamento.
- `input_example.json`: exemplo concreto de entrada de dados válida. Serve para documentação, testes e validação, ajudando a entender quais dados o modelo espera receber.
- `model.pkl`: artefato do modelo serializado. É aqui que reside a lógica matemática do modelo treinado que será carregada na memória para realizar as predições.
- `python_env.yaml`: metadados do ambiente Python, especificando a versão do Python e as dependências que devem estar presentes para que o código do modelo seja executado corretamente.
- `registered_model_meta`: metadados específicos do registro do modelo no **MLflow Model Registry**. Armazena informações sobre o nome do modelo, a versão, o alias e outros detalhes de governança do ciclo de vida do modelo.
- `requirements.txt`: lista das bibliotecas e suas respectivas versões necessárias para a execução. Alternativa ao `conda.yaml` para ambientes que utilizam gerenciamento via `pip`.
- `serving_input_example.json`: versão do exemplo de entrada formatada especificamente para o serviço de inferência (_serving_). Facilita a configuração correta do _payload_ quando o modelo é colocado em produção via API.

### 1.3 Arquivo `MLmodel`

In [15]:
mlmodel_path = local_model_path / "MLmodel"
print(mlmodel_path.read_text(encoding="utf-8"))

artifact_path: /home/bruno/estudos/data-science-portifolio/mlflow/mlartifacts/5/models/m-493132b1b8b64441bb51b13a29ab3fe3/artifacts
flavors:
  python_function:
    env:
      conda: conda.yaml
      virtualenv: python_env.yaml
    loader_module: mlflow.sklearn
    model_path: model.pkl
    predict_fn: predict
    python_version: 3.12.3
  sklearn:
    code: null
    pickled_model: model.pkl
    serialization_format: cloudpickle
    sklearn_version: 1.9.1
    skops_trusted_types: true
is_signature_from_type_hint: false
mlflow_version: 3.17.0
model_id: m-493132b1b8b64441bb51b13a29ab3fe3
model_size_bytes: 4837
model_uuid: m-493132b1b8b64441bb51b13a29ab3fe3
prompts: null
run_id: d8ad51b2a7a94177a2ce086c8df64e7b
saved_input_example_info:
  artifact_path: input_example.json
  pandas_orient: split
  serving_input_path: serving_input_example.json
  type: dataframe
signature:
  inputs: '[{"type": "long", "name": "idade", "required": true}, {"type": "double",
    "name": "renda_mensal", "required

In [ ]:
# abrindo o conteudo do arquivo em formato de dicionario
import yaml
import json

with open(mlmodel_path, encoding="utf-8") as f:
    mlmodel_config = yaml.safe_load(f)

print(json.dumps(mlmodel_config, indent=4))

{
    "artifact_path": "/home/bruno/estudos/data-science-portifolio/mlflow/mlartifacts/5/models/m-493132b1b8b64441bb51b13a29ab3fe3/artifacts",
    "flavors": {
        "python_function": {
            "env": {
                "conda": "conda.yaml",
                "virtualenv": "python_env.yaml"
            },
            "loader_module": "mlflow.sklearn",
            "model_path": "model.pkl",
            "predict_fn": "predict",
            "python_version": "3.12.3"
        },
        "sklearn": {
            "code": null,
            "pickled_model": "model.pkl",
            "serialization_format": "cloudpickle",
            "sklearn_version": "1.9.1",
            "skops_trusted_types": true
        }
    },
    "is_signature_from_type_hint": false,
    "mlflow_version": "3.17.0",
    "model_id": "m-493132b1b8b64441bb51b13a29ab3fe3",
    "model_size_bytes": 4837,
    "model_uuid": "m-493132b1b8b64441bb51b13a29ab3fe3",
    "prompts": null,
    "run_id": "d8ad51b2a7a94177a2ce086c8df6

### 1.4 _Flavors_

Os _flavors_ são uma das características mais poderosas do MLflow. Eles servem como diferentes "interfaces" para o mesmo modelo, permitindo que ele seja carregado de maneiras distintas, dependendo da necessidade da aplicação. Ter mais de um _flavor_ é interessante por alguns motivos principais:

- Flexibilidade no consumo: ao treinar um modelo com o Scikit-Learn, o _flavor_ nativo (`sklearn`) permite carregar o objeto exatamente como ele é, o que é bom quando é necessário acessar métodos específicos do _framework_, como o `predict_proba()` ou acessar atributos internos do estimador.
- Padronização com o Python Function (PyFunc): O _flavor_ `python_function` é a camada de abstração universal do MLflow. Quando um modelo é carregado por meio desse _flavor_, o MLflow o trata como uma função genérica com um método `predict()` padrão. Isso é bom porque, para o sistema de produção ou a ferramenta de _serving_, não importa se o modelo foi feito em Scikit-Learn, XGBoost, PyTorch ou TensorFlow, a aplicação consumidora fala apenas a "língua" do pyfunc, o que torna o código de _deploy_ agnóstico ao _framework_ original.
- Desacoplamento: Graças aos múltiplos _flavors_, a aplicação de produção não precisa ter todas as bibliotecas pesadas de treinamento instaladas, nem precisa saber como instanciar o objeto original do framework. O _flavor_ `pyfunc` gerencia isso internamente, garantindo que o comportamento seja consistente através de uma interface única.

Resumindo, ter múltiplos _flavors_ é como ter um adaptador universal. O _flavor_ nativo fornece todo o poder e especificidade da biblioteca de origem, enquanto o _flavor_ `pyfunc` possui a simplicidade e portabilidade necessárias para o modelo funcionar em qualquer ambiente de produção, sem causar erros de compatibilidade ou dependências complexas.

In [27]:
print(json.dumps(mlmodel_config.get("flavors"), indent=4))

{
    "python_function": {
        "env": {
            "conda": "conda.yaml",
            "virtualenv": "python_env.yaml"
        },
        "loader_module": "mlflow.sklearn",
        "model_path": "model.pkl",
        "predict_fn": "predict",
        "python_version": "3.12.3"
    },
    "sklearn": {
        "code": null,
        "pickled_model": "model.pkl",
        "serialization_format": "cloudpickle",
        "sklearn_version": "1.9.1",
        "skops_trusted_types": true
    }
}


Abaixo carregamos o modelo usando os dois _flavors_ disponíveis, primeiro com `sklearn` e depois com `pyfunc`:

In [ ]:
# carrega modelo usando flavor sklearn
import sys
sys.path.insert(0, "../src")

import mlflow.sklearn
from support_module import profile_for_test_prediction

sklearn_model = mlflow.sklearn.load_model(production_model_uri)
novo_cliente  = profile_for_test_prediction()

classe        = sklearn_model.predict(novo_cliente)[0]
probabilidade = sklearn_model.predict_proba(novo_cliente)[0, 1]

print("Classe prevista:", classe)
print(f"Probabilidade de inadimplência: {probabilidade:.1%}")

Classe prevista: 0
Probabilidade de inadimplência: 27.2%


In [ ]:
# carrega modelo usadno flavor pyfunc
pyfunc_model = mlflow.pyfunc.load_model(production_model_uri)
predicao     = pyfunc_model.predict(novo_cliente)

predicao # por padrao, temos acesso apenas a classe resultante da predicao

array([0])

### 1.5 _Signature_

A _signature_ funciona como um contrato formal entre o seu modelo e o mundo exterior, é crucial principalmente em ambientes de produção, onde a previsibilidade é essencial. Ela serve para definir, de forma explícita e documentada, exatamente quais são os _inputs_ que o modelo espera receber e qual o _output_ será devolvido. 

Dentre os princípais pontos temos:

- Validação estrutural: a _signature_ define quais colunas, nomes e tipos de dados (como inteiros, pontos flutuantes ou strings) são necessários. Quando um dado é enviado para o modelo, o MLflow utiliza essa assinatura para validar se o que está chegando é compatível com o que foi treinado. Se faltar uma coluna ou o tipo de dado estiver errado, o modelo pode disparar um erro antes mesmo de tentar processar, evitando resultados imprevisíveis.
- Prevenção de erros silenciosos: sem uma assinatura, o modelo poderia receber uma coluna com nome ligeiramente diferente ou em ordem diferente do esperado, e ainda assim rodar e gerar uma predição sem sentido, mas que o sistema aceitaria o resultado como válido. Com a assinatura, o "contrato" impede que esse tipo de falha silenciosa ocorra, garantindo que o modelo sempre opere dentro dos parâmetros esperados.
- Facilitação do _deployment_: ao disponibilizar o modelo como API, a assinatura ajuda a gerar automaticamente a documentação do _endpoint_. Ferramentas de servir o modelo usam essas informações para entender como formatar o JSON de entrada, reduzindo drasticamente o trabalho de configuração manual e evitando erros ao integrar o modelo com outras aplicações ou serviços.
- Documentação técnica: serve como um registro claro e técnico do que compõe a interface do modelo, qualquer outro desenvolvedor que for consumir seu modelo, saberá exatamente qual _payload_ precisa enviar.

Resumindo, a assinatura é a garantia de que o modelo não será "surpreendido" por dados incompatíveis.

In [40]:
from mlflow.models import get_model_info

model_info = get_model_info(production_model_uri)

print(model_info.signature)

inputs: 
  ['idade': long (required), 'renda_mensal': double (required), 'tempo_emprego_anos': double (required), 'score_interno': long (required), 'valor_solicitado': double (required), 'num_parcelas': long (required), 'tipo_renda': string (required), 'possui_restricao': string (required)]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None



Abaixo, vamos testar algumas entradas válidas e não válidas:

In [46]:
from support_module import RANDOM_STATE, generate_credit_risk_base
from sklearn.model_selection import train_test_split

# gera base de dados sintetica para risco de credito
df = generate_credit_risk_base(6000)
print(df.shape)

# seleciona features
TARGET   = "inadimplente"
FEATURES = [c for c in df.columns if c != TARGET]
x        = df[FEATURES]
y        = df[TARGET]

# separa colunas numericas e categoricas
num_cols = df[FEATURES].select_dtypes(include='number').columns
cat_cols = [f for f in FEATURES if f not in num_cols]

# separa dados de treino e teste, seguindo proporcao de casos positivos e negativos do target
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

print(x_train.shape, x_test.shape)

(6000, 9)
(4500, 8) (1500, 8)


In [48]:
# testa predicao de uma entrada valida com 3 registros
entrada_valida = x_test.head(3).copy()

pyfunc_model.predict(entrada_valida)

array([0, 0, 0])

In [52]:
# testa predicao com entrada invalida: removendo uma coluna
entrada_sem_coluna = entrada_valida.drop(columns=["score_interno"])

try:
    pyfunc_model.predict(entrada_sem_coluna)
except Exception as e:
    print(type(e).__name__)
    print(e)

MlflowException
Failed to enforce schema of data '      idade  renda_mensal  tempo_emprego_anos  valor_solicitado  num_parcelas  \
4939     36       3462.10                 6.8          12499.99            36   
2501     26       2294.28                 0.7           7903.58            36   
145      59       1986.35                 2.4           8163.56            48   

     tipo_renda possui_restricao  
4939   autonomo              nao  
2501        CLT              nao  
145    autonomo              nao  ' with schema '['idade': long (required), 'renda_mensal': double (required), 'tempo_emprego_anos': double (required), 'score_interno': long (required), 'valor_solicitado': double (required), 'num_parcelas': long (required), 'tipo_renda': string (required), 'possui_restricao': string (required)]'. Error: Model is missing inputs ['score_interno'].


In [53]:
# testa predicao com entrada invalida: adicionando coluna extra
entrada_com_coluna_extra = entrada_valida.copy()
entrada_com_coluna_extra["canal_origem"] = "app"

pyfunc_model.predict(entrada_com_coluna_extra)

2026/10/18 12:58:41 WARNING mlflow.models.utils: Found extra inputs in the model input that are not defined in the model signature: `['canal_origem']`. These inputs will be ignored.


array([0, 0, 0])

Os arquivos que contém as informações da _signature_ são:

- `'input_example.json'`: exemplo de entrada bruto, referência dos dados usados no treinamento
- `'serving_input_example.json'`: versão otimizada para o serviço de inferência, reflete exatamente o formato que o serviço disponibilizado pelo MLflow espera receber ao processar uma requisição de predição.

In [61]:
input_example_path = local_model_path / "input_example.json"
print(
    json.dumps(
        json.loads(input_example_path.read_text(encoding="utf-8")), 
        indent=4
    )
)

{
    "columns": [
        "idade",
        "renda_mensal",
        "tempo_emprego_anos",
        "score_interno",
        "valor_solicitado",
        "num_parcelas",
        "tipo_renda",
        "possui_restricao"
    ],
    "data": [
        [
            62,
            5669.6,
            5.1,
            772,
            5284.94,
            24,
            "autonomo",
            "nao"
        ],
        [
            65,
            2862.53,
            6.4,
            521,
            9535.78,
            36,
            "autonomo",
            "nao"
        ],
        [
            65,
            4789.26,
            0.1,
            691,
            9327.15,
            12,
            "CLT",
            "nao"
        ],
        [
            42,
            5216.54,
            5.6,
            577,
            8358.31,
            48,
            "aposentado",
            "nao"
        ],
        [
            45,
            2251.25,
            3.9,
            743,
  

In [62]:
serving_example_path = local_model_path / "serving_input_example.json"

print(
    json.dumps(
        json.loads(serving_example_path.read_text(encoding="utf-8")), 
        indent=4
    )
)

{
    "dataframe_split": {
        "columns": [
            "idade",
            "renda_mensal",
            "tempo_emprego_anos",
            "score_interno",
            "valor_solicitado",
            "num_parcelas",
            "tipo_renda",
            "possui_restricao"
        ],
        "data": [
            [
                62,
                5669.6,
                5.1,
                772,
                5284.94,
                24,
                "autonomo",
                "nao"
            ],
            [
                65,
                2862.53,
                6.4,
                521,
                9535.78,
                36,
                "autonomo",
                "nao"
            ],
            [
                65,
                4789.26,
                0.1,
                691,
                9327.15,
                12,
                "CLT",
                "nao"
            ],
            [
                42,
                5216.54,
        

## 2 Customizando o `pyfunc`

Em algumas situações, saber apenas a classe predita (que é comportamento padrão do algoritmo) não é suficiente. Nesses casos, podemos usar o flavor `pyfunc` do MLflow para criar um modelo personalizado, encapsulando dentro dele não apenas o algoritmo de _machine learning_, mas também toda a lógica de pré-processamento, pós-processamento, validação de dados, regras de negócio e até múltiplos artefatos necessários para a inferência. Ou seja, podemos empacotar o modelo com uma lógica de inferência mais adequada para seu uso por aplicações.

Dessa forma estaremos transformando o modelo em uma classe autocontida, e independentemente da biblioteca original usada para treinar o modelo, a aplicação que for consumi-lo interaja sempre através de uma interface padronizada. Isso garante que a lógica de negócio necessária para interpretar o resultado (como aplicar um limiar ou tratar colunas faltantes) seja executada automaticamente pelo próprio modelo, e não pela aplicação que o consome.

### 2.1 Classe customizada do PyFunc

Pensando no modelo para análise de risco de crédito, que hoje nos entrega apenas a classe prevista, isso vai possibilitar entregar respostas mais ricas e interpretáveis, como probabilidades e limiares da decisão. Começamos carregando os arquivos do modelo do Registry:

In [65]:
production_model_uri

'models:/credit_risk_classifier@production'

In [ ]:
# traz arquivos do modelo para um dir temp local na maquina, mas como ja estao aqui, nao ha necessidade e apenas retorna o path de onde estao
sklearn_model_local_path = mlflow.artifacts.download_artifacts(artifact_uri=production_model_uri)
sklearn_model_local_path

'/home/bruno/estudos/data-science-portifolio/mlflow/mlartifacts/5/models/m-493132b1b8b64441bb51b13a29ab3fe3/artifacts'

Para a criação de modelos customizados com `mlflow.pyfunc`, o MLflow impõe uma assinatura de interface, o que significa que seus métodos precisam aceitar os parâmetros que o MLflow espera, caso contrário, o motor de execução do modelo falhará ao tentar chamá-los. No método `predict`, por exemplo, o MLflow espera os seguintes argumentos:

- `context`: é o objeto que carrega os artefatos, seu tipo correto é `mlflow.pyfunc.PythonModelContext`;
- `model_input`: os dados de _input_ podem variar de formato, chegando como DataFrame, dicionário ou lista, por isso é uma boa prática garantir a conversão para `pd.DataFrame`;
- `params`: o retorno deve ser um formato que a aplicação cliente saiba interpretar, geralmente um DataFrame ou lista de valores.

O uso _Type Hints_ não é obrigatório, mas muito incentivado, pois torna o contrato de dados do projeto muito mais claro para outros desenvolvedores. 

Além do `predict` temos o método `load_context`, que é um método "_hook_". O MLflow o invoca automaticamente sempre que o modelo é carregado através da função `mlflow.pyfunc.load_model`. O `load_context` também recebe um objeto chamado `context`, um contentor que o próprio MLflow preenche (acessando o arquivo MLmodel) com os caminhos dos artefatos definidos no momento do _log_ do modelo (dicionário _artifacts_). Esse método que se "aproveita" do _path_ retornado acima para carregar o modelo de volta na memória durante a inferência.

Abaixo a classe customizada:

In [75]:
import numpy as np
import pandas as pd
import mlflow.pyfunc
import mlflow.sklearn
from typing import Dict, Any, Optional


class CreditRiskPyfuncModel(mlflow.pyfunc.PythonModel):
    def __init__(self, required_columns, default_threshold=0.5):
        self.required_columns  = list(required_columns)
        self.default_threshold = float(default_threshold)

    def load_context(self, context: mlflow.pyfunc.PythonModelContext) -> None:
        self.sklearn_model = mlflow.sklearn.load_model(context.artifacts["sklearn_model"])

    def predict(
        self, 
        context: mlflow.pyfunc.PythonModelContext, 
        model_input: pd.DataFrame, 
        params: Optional[Dict[str, Any]] = None
    ) -> pd.DataFrame:
        if not isinstance(model_input, pd.DataFrame):
            model_input = pd.DataFrame(model_input)

        # verifica se existem colunas faltantes no input
        missing_columns = [col for col in self.required_columns if col not in model_input.columns]

        if missing_columns:
            raise ValueError(f"Colunas obrigatórias ausentes: {missing_columns}")

        threshold = self.default_threshold

        # sobreescreve limiar se tiver passado um novo valor em params
        if params is not None and "threshold" in params:
            threshold = float(params["threshold"])

        # faz predicao
        x                 = model_input[self.required_columns].copy() # seleciona apenas colunas obrigatorias
        probabilities     = self.sklearn_model.predict_proba(x)[:, 1] # calcula probabilidades
        predicted_classes = (probabilities >= threshold).astype(int)  # determina classes previstas

        return pd.DataFrame({
            "probabilidade_inadimplencia": probabilities,
            "classe_prevista": predicted_classes,
            "limiar_utilizado": np.repeat(threshold, len(x)),
        })



Antes de "_loggar_" (registrar) a classe, é sempre bom testar:

In [76]:
wrapper_teste = CreditRiskPyfuncModel(
    required_columns=FEATURES,
    default_threshold=0.5,
)

wrapper_teste.sklearn_model = mlflow.sklearn.load_model(production_model_uri)
input_example               = x_train.head(5).copy()
output_example              = wrapper_teste.predict(context=None, model_input=input_example)

output_example

,probabilidade_inadimplencia,classe_prevista,limiar_utilizado
0,0.010761,0,0.5
1,0.914754,1,0.5
2,0.063973,0,0.5
3,0.464377,0,0.5
4,0.031858,0,0.5


Como alteramos a saída do modelo (adicionamos as colunas probabilidade e limiar a classe prevista), precisamos "atualizar" seu contrato de dados, para que ele descreva exatamente quais são as colunas de entrada e o formato exato de saída que o modelo produzirá. Note que o contrato de dados passa a aceitar um limiar de predição customizado, ou seja, será possível definir o limiar no momento que formos fazer a predição do modelo.

In [ ]:
from mlflow.models import infer_signature

signature_for_custom_class = infer_signature(
    input_example,            
    output_example,           # note que passamos os exemplos de input e output usados no teste acima
    params={"threshold": 0.5} # modelo tambem passa a aceitar um novo limiar como parametro de configuracao
)
signature_for_custom_class

/home/bruno/estudos/data-science-portifolio/mlflow/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


inputs: 
  ['idade': long (required), 'renda_mensal': double (required), 'tempo_emprego_anos': double (required), 'score_interno': long (required), 'valor_solicitado': double (required), 'num_parcelas': long (required), 'tipo_renda': string (required), 'possui_restricao': string (required)]
outputs: 
  ['probabilidade_inadimplencia': double (required), 'classe_prevista': long (required), 'limiar_utilizado': double (required)]
params: 
  ['threshold': double (default: 0.5)]


### 2.2 Registrando a classe customizada

Agora vamos registrar ("_loggar_") o modelo customizado, dessa forma, ele poderá ser carregado como qualquer outro modelo do MLflow, mas com iterface de predição customizada.

In [83]:
PYFUNC_MODEL_NAME = "credit_risk_classifier_pyfunc"

mlflow.set_experiment("risco_credito_custom_pyfunc")

with mlflow.start_run(run_name="register_custom_pyfunc_credit_risk") as run:
    pyfunc_model_info = mlflow.pyfunc.log_model(
        name="credit_risk_pyfunc",
        python_model=CreditRiskPyfuncModel(
            required_columns=FEATURES,
            default_threshold=0.5,
        ),
        artifacts={
            "sklearn_model": sklearn_model_local_path # path dos arquivos do modelo: '(...)/mlartifacts/5/models/m-493132b1b8b64441bb51b13a29ab3fe3/artifacts'
        },
        signature=signature_for_custom_class,
        input_example=input_example,
        registered_model_name=PYFUNC_MODEL_NAME,
        pip_requirements=[
            "mlflow",
            "pandas",
            "numpy",
            "scikit-learn",
            "cloudpickle",
        ],
    )

    mlflow.log_params({
        "source_model_uri":      production_model_uri, # 'models:/credit_risk_classifier@production'
        "default_threshold":     0.5,
        "output_format":         "probability_label_threshold",
        "registered_model_name": PYFUNC_MODEL_NAME,
    })

    mlflow.set_tags({
        "project":         "risco_credito",
        "model_interface": "custom_pyfunc",
        "run_purpose":     "custom_inference_contract",
    })

    pyfunc_run_id = run.info.run_id

print("Run ID:", pyfunc_run_id)
print("Model URI:", pyfunc_model_info.model_uri)

2026/10/18 16:57:08 INFO mlflow.tracking.fluent: Experiment with name 'risco_credito_custom_pyfunc' does not exist. Creating a new experiment.
2026/10/18 16:57:11 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/10/18 16:57:11 INFO mlflow.pyfunc: Validating input example against model signature
Successfully registered model 'credit_risk_classifier_pyfunc'.
2026/10/18 16:57:13 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: credit_risk_classifier_pyfunc, version 1
Created version '1' of model 'credit_risk_classifier_pyfunc'.


🏃 View run register_custom_pyfunc_credit_risk at: http://127.0.0.1:5000/#/experiments/6/runs/bdcaa76259564975aa075b83915f69b5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6
Run ID: bdcaa76259564975aa075b83915f69b5
Model URI: models:/m-9bc8a1fd3bbe4b4f9045c921ed1fa36b


Ao utilizar o método `mlflow.pyfunc.log_model`, o MLflow cria uma nova estrutura de diretórios e arquivos dentro da pasta de artefatos em `mlartifacts/6/models/m-9bc8a1fd3bbe4b4f9045c921ed1fa36b`.

Note que como foi definido definido o modelo Scikit-Learn como um "artefato base" (`artifacts={"sklearn_model": sklearn_model_local_path}`), o MLflow copia esse arquivo do modelo original, garantindo que o modelo customizado seja "autossuficiente", tendo consigo tudo o que precisa para rodar.

Agora que temos o modelo customizado registrado, podemos carregar e testar:

In [ ]:
# carrega modelo customizado usando URI gerada no registro e testa predicao
loaded_pyfunc = mlflow.pyfunc.load_model(pyfunc_model_info.model_uri)

loaded_pyfunc.predict(novo_cliente)

,probabilidade_inadimplencia,classe_prevista,limiar_utilizado
0,0.272417,0,0.5


In [85]:
# testa predicao definindo um limiar diferente
loaded_pyfunc.predict(novo_cliente, params={"threshold": 0.25})

,probabilidade_inadimplencia,classe_prevista,limiar_utilizado
0,0.272417,1,0.25


In [89]:
# testa predicao passando input com uma coluna a menos
cliente_invalido = novo_cliente.drop(columns=["score_interno"])

try:
    loaded_pyfunc.predict(cliente_invalido)
except Exception as e:
    print(type(e).__name__)
    print(e)

MlflowException
Failed to enforce schema of data '   idade  renda_mensal  tempo_emprego_anos  valor_solicitado  num_parcelas  \
0     34        4200.0                 5.0           12000.0            24   

  tipo_renda possui_restricao  
0        CLT              nao  ' with schema '['idade': long (required), 'renda_mensal': double (required), 'tempo_emprego_anos': double (required), 'score_interno': long (required), 'valor_solicitado': double (required), 'num_parcelas': long (required), 'tipo_renda': string (required), 'possui_restricao': string (required)]'. Error: Model is missing inputs ['score_interno'].


### 2.3 Definindo _aliases_ para a classe customizada

In [90]:
from mlflow import MlflowClient

client = MlflowClient()

pyfunc_versions = client.search_model_versions(
    f"name = '{PYFUNC_MODEL_NAME}'"
)

pyfunc_registered_version = max(
    [mv for mv in pyfunc_versions if mv.run_id == pyfunc_run_id],
    key=lambda mv: int(mv.version)
)

# atribui aliases indicando que a versao eh candidata, e no caso, candidata a serving (ficar acessivel para ser consumido via API, por exemplo)
client.set_registered_model_alias(name=PYFUNC_MODEL_NAME, version=pyfunc_registered_version.version, alias="candidate")
client.set_registered_model_alias(name=PYFUNC_MODEL_NAME, version=pyfunc_registered_version.version, alias="serving_candidate")
client.set_model_version_tag(PYFUNC_MODEL_NAME, pyfunc_registered_version.version, "interface", "probability_label_threshold")

print("Versão pyfunc:", pyfunc_registered_version.version)

Versão pyfunc: 1


Após atribuir _aliases_ ao modelo, podemos carregá-lo do Registry usando um desses _aliases_:

In [91]:
serving_candidate_uri   = f"models:/{PYFUNC_MODEL_NAME}@serving_candidate"
serving_candidate_model = mlflow.pyfunc.load_model(serving_candidate_uri)

serving_candidate_model.predict(novo_cliente)

,probabilidade_inadimplencia,classe_prevista,limiar_utilizado
0,0.272417,0,0.5


In [92]:
serving_candidate_model.predict(novo_cliente, params={"threshold": 0.25})

,probabilidade_inadimplencia,classe_prevista,limiar_utilizado
0,0.272417,1,0.25
